## Cell 1: Setup
**What this demonstrates**: Two-model configuration — `claude-haiku` for fast relevance critiques (Yes/No judgments), `claude-sonnet` for generation and answer quality critique. The critique functions return structured verdicts so every decision can be logged.
**Expected output**: ✓ Setup complete with both model names, embed model, and masked key suffixes.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
%pip install -q langchain langchain-openai langchain-community chromadb python-dotenv

# ── Standard library ─────────────────────────────────────────────────────────
import os
import time
import pathlib
from dataclasses import dataclass, field

# ── Third-party ──────────────────────────────────────────────────────────────
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

# ── Model constants ───────────────────────────────────────────────────────────
# Haiku for critique: binary Yes/No judgments — speed matters more than nuance
CRITIQUE_MODEL = 'claude-haiku-4-5-20251001'
# Sonnet for generation and answer quality critique — reasoning quality matters here
ANSWER_MODEL   = 'claude-sonnet-4-6'
EMBED_MODEL    = 'text-embedding-3-small'
CHROMA_DIR     = './chroma_self_rag'
CHUNK_SIZE     = 400

# ── Clients ───────────────────────────────────────────────────────────────────
client:     Anthropic        = Anthropic()
embeddings: OpenAIEmbeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# ── Critique result dataclass ────────────────────────────────────────────────
# Stores the verdict (Yes/No) and rationale for each critique call.
# In production: persist alongside the answer for audit and debugging.
@dataclass
class CritiqueResult:
    verdict:   str   # 'Yes' or 'No'
    rationale: str   # one-sentence explanation

    @property
    def passed(self) -> bool:
        return self.verdict.lower().startswith('yes')


print('✓ Setup complete')
print(f'  Critique model: {CRITIQUE_MODEL}')
print(f'  Answer model:   {ANSWER_MODEL}')
print(f'  Embed model:    {EMBED_MODEL}')
print(f'  Anthropic key:  ...{_anthropic_key[-4:]}')
print(f'  OpenAI key:     ...{_openai_key[-4:]}')
print()
print('Self-RAG critique loop:')
print('  1. Retrieve top-5 chunks')
print('  2. Critique each chunk: is it relevant? (Yes / No)')
print('  3. Filter to relevant chunks only')
print('  4. Generate answer from filtered chunks')
print('  5. Critique answer: is it accurate and complete? (Yes / No)')
print('  6. If No: regenerate or return "Cannot answer with confidence"')

## Cell 2: Data — Basel III Regulatory Excerpt
**What this demonstrates**: Index the Basel III regulatory document — a corpus with multiple distinct topics (capital ratios, leverage, liquidity, stress testing). A query about LCR requirements will retrieve chunks from different sections, giving the ISREL critique step meaningful filtering work to do.
**Expected output**: Chunk count per article section, and a preview of the LCR-relevant content that the query should surface.

In [ ]:
# ── Locate document ───────────────────────────────────────────────────────────
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data.'

doc_text = (BASE_DIR / 'basel_iii_excerpt.txt').read_text(encoding='utf-8')

# ── Chunk at section boundaries then by size ──────────────────────────────────
# Primary separator: the section divider used in the document
# Secondary: paragraph breaks, then line breaks
splitter = RecursiveCharacterTextSplitter(
    separators=['\n================================================================================\n', '\n\n', '\n', ' '],
    chunk_size=CHUNK_SIZE,
    chunk_overlap=40,
)
raw_chunks = splitter.split_text(doc_text)

all_docs: list[Document] = [
    Document(page_content=chunk, metadata={'chunk_idx': i})
    for i, chunk in enumerate(raw_chunks)
]

# ── Build vector store ────────────────────────────────────────────────────────
vectorstore = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    collection_name='self_rag_basel',
    persist_directory=CHROMA_DIR,
)

print(f'Document: basel_iii_excerpt.txt  ({len(doc_text):,} chars  →  {len(all_docs)} chunks)')
print()

# Identify which chunks contain LCR content — shows what ISREL should accept
lcr_chunks = [d for d in all_docs if 'lcr' in d.page_content.lower() or 'liquidity coverage' in d.page_content.lower()]
other_chunks = len(all_docs) - len(lcr_chunks)

print(f'Chunks containing LCR content:   {len(lcr_chunks)}')
print(f'Chunks from other sections:       {other_chunks}')
print()
print('Why this tests Self-RAG well:')
print('  A query about LCR will retrieve a mix of LCR chunks AND chunks about')
print('  capital ratios, leverage, NSFR, or stress testing that score reasonably')
print('  on cosine similarity. The ISREL critique step must filter those out.')
print()
print('LCR-relevant content preview:')
if lcr_chunks:
    preview = lcr_chunks[0].page_content[:200].replace('\n', ' ')
    print(f'  {preview!r}...')

## Cell 3: Core — Self-RAG Critique Loop
**What this demonstrates**: The complete Self-RAG pipeline as three functions: `critique_relevance` (is this chunk relevant? Yes/No), `generate_answer` (answer from filtered chunks only), and `critique_answer` (is this accurate and complete? Yes/No). A fourth function `self_rag` wires them into the full loop with the fallback path.
**Expected output**: Function definitions confirmed, then a live test of `critique_relevance` on one relevant and one irrelevant chunk.

In [ ]:
# ── Critique 1: Relevance — is this chunk relevant to the query? ──────────────
def critique_relevance(query: str, chunk: Document) -> CritiqueResult:
    """Judge whether a retrieved chunk is relevant to the query.

    Uses Haiku for speed — this is a binary judgment, not a reasoning task.
    A chunk must directly address the question to pass; tangentially related
    content is marked No and filtered out before generation.

    Args:
        query: The user's question.
        chunk: A retrieved Document from the vector store.

    Returns:
        CritiqueResult with verdict 'Yes' or 'No' and a one-sentence rationale.
    """
    response = client.messages.create(
        model=CRITIQUE_MODEL,
        max_tokens=80,
        system=(
            'You decide whether a document excerpt is relevant to a question. '
            'Answer Yes if the excerpt directly addresses the question — contains '
            'specific facts, definitions, requirements, or thresholds the question asks about. '
            'Answer No if the excerpt addresses a different topic, even if from the same domain. '
            'Respond in exactly this format:\n'
            'Verdict: Yes or No\n'
            'Reason: one sentence'
        ),
        messages=[{'role': 'user', 'content': (
            f'Question: {query}\n\n'
            f'Excerpt:\n{chunk.page_content[:500]}'
        )}],
    )
    text = response.content[0].text.strip()
    verdict  = 'Yes' if 'Verdict: Yes' in text else 'No'
    rationale = text.split('Reason:', 1)[-1].strip() if 'Reason:' in text else text
    return CritiqueResult(verdict=verdict, rationale=rationale)


# ── Generate: produce answer from relevant chunks only ─────────────────────────
def generate_answer(query: str, relevant_chunks: list[Document]) -> str:
    """Generate an answer using only the chunks that passed the relevance critique.

    Args:
        query:           The user's original question.
        relevant_chunks: Documents that passed critique_relevance (Yes verdict).

    Returns:
        A grounded answer citing specific requirements from the provided excerpts.
    """
    context = '\n\n---\n\n'.join(
        f'[Excerpt {i+1}]\n{doc.page_content}'
        for i, doc in enumerate(relevant_chunks)
    )
    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=400,
        system=(
            'You are a regulatory compliance analyst. '
            'Answer the question using ONLY the provided document excerpts. '
            'State each requirement or threshold as a separate sentence. '
            'Cite the excerpt number for every specific figure or rule. '
            'Do not infer or add information not present in the excerpts.'
        ),
        messages=[{'role': 'user', 'content': f'Excerpts:\n{context}\n\nQuestion: {query}'}],
    )
    return response.content[0].text.strip()


# ── Critique 2: Answer quality — is the answer accurate and complete? ─────────
def critique_answer(query: str, answer: str, relevant_chunks: list[Document]) -> CritiqueResult:
    """Verify whether the generated answer is accurate and complete.

    Checks two things: (1) every claim in the answer is supported by the
    provided excerpts, and (2) the answer actually addresses the question.
    Uses Sonnet — this requires careful reasoning about claim-document alignment.

    Args:
        query:           The original question.
        answer:          The generated answer to critique.
        relevant_chunks: The source documents used to generate the answer.

    Returns:
        CritiqueResult — Yes if the answer is grounded and complete; No otherwise.
    """
    context = '\n---\n'.join(doc.page_content[:400] for doc in relevant_chunks)
    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=100,
        system=(
            'You verify whether an answer is accurate and complete given source documents. '
            'Answer Yes if: (1) every specific claim in the answer is supported by the documents, '
            'and (2) the answer addresses the question asked. '
            'Answer No if: any claim contradicts or is absent from the documents, '
            'or the answer is too vague to be useful. '
            'Respond in exactly this format:\n'
            'Verdict: Yes or No\n'
            'Reason: one sentence'
        ),
        messages=[{'role': 'user', 'content': (
            f'Question: {query}\n\n'
            f'Source documents:\n{context}\n\n'
            f'Answer to verify:\n{answer}'
        )}],
    )
    text = response.content[0].text.strip()
    verdict   = 'Yes' if 'Verdict: Yes' in text else 'No'
    rationale = text.split('Reason:', 1)[-1].strip() if 'Reason:' in text else text
    return CritiqueResult(verdict=verdict, rationale=rationale)


# ── Full Self-RAG loop ────────────────────────────────────────────────────────
def self_rag(
    query:    str,
    k:        int = 5,
    max_retries: int = 1,
) -> dict:
    """Run the Self-RAG critique loop.

    Flow:
        1. Retrieve top-k chunks
        2. Critique each chunk for relevance — keep Yes verdicts only
        3. If no chunks pass: return cannot-answer response
        4. Generate answer from relevant chunks
        5. Critique answer quality
        6. If No: retry once (with same chunks) or return cannot-answer

    Args:
        query:       The user's question.
        k:           Number of chunks to retrieve.
        max_retries: How many times to regenerate on a failed answer critique.

    Returns:
        dict with keys: answer, retrieved, relevance_critiques,
        relevant_chunks, answer_critique, final_status
    """
    # Step 1: retrieve
    retrieved = vectorstore.similarity_search(query, k=k)

    # Step 2: critique each chunk for relevance
    relevance_critiques: list[tuple[Document, CritiqueResult]] = []
    for chunk in retrieved:
        critique = critique_relevance(query, chunk)
        relevance_critiques.append((chunk, critique))

    # Step 3: filter to relevant chunks only
    relevant_chunks = [doc for doc, cr in relevance_critiques if cr.passed]

    if not relevant_chunks:
        return {
            'answer': 'Cannot answer with confidence: no retrieved documents were relevant to this query.',
            'retrieved': retrieved,
            'relevance_critiques': relevance_critiques,
            'relevant_chunks': [],
            'answer_critique': None,
            'final_status': 'abstained — no relevant chunks',
        }

    # Step 4 + 5: generate then critique, with retry
    answer = generate_answer(query, relevant_chunks)
    answer_critique = critique_answer(query, answer, relevant_chunks)

    for _ in range(max_retries):
        if answer_critique.passed:
            break
        # Retry: regenerate with the same relevant chunks
        answer = generate_answer(query, relevant_chunks)
        answer_critique = critique_answer(query, answer, relevant_chunks)

    # Step 6: if still failing after retries, return cannot-answer
    if not answer_critique.passed:
        answer = 'Cannot answer with confidence: the available documents do not support a complete answer to this question.'
        final_status = 'abstained — answer failed quality critique'
    else:
        final_status = 'delivered'

    return {
        'answer': answer,
        'retrieved': retrieved,
        'relevance_critiques': relevance_critiques,
        'relevant_chunks': relevant_chunks,
        'answer_critique': answer_critique,
        'final_status': final_status,
    }


# ── Smoke test: critique_relevance on one relevant and one irrelevant chunk ────
print('Smoke test — critique_relevance on two contrasting chunks:')
print()

test_query = 'What are the liquidity coverage ratio requirements?'

# Find a chunk that should be relevant (contains LCR text)
lcr_test = next((d for d in all_docs if 'liquidity coverage' in d.page_content.lower()), None)
# Find a chunk that should not be relevant (capital ratios, not liquidity)
cap_test = next((d for d in all_docs if 'cet1' in d.page_content.lower() and 'lcr' not in d.page_content.lower()), None)

for label, doc in [('LCR section', lcr_test), ('Capital ratio section', cap_test)]:
    if doc:
        result = critique_relevance(test_query, doc)
        marker = '✓ kept' if result.passed else '✗ filtered'
        print(f'  [{label}]')
        print(f'  Verdict: {result.verdict}  {marker}')
        print(f'  Reason:  {result.rationale}')
        print(f'  Content: {doc.page_content[:80].replace(chr(10), " ")}...')
        print()

## Cell 4: Run — LCR Requirements Query
**What this demonstrates**: End-to-end Self-RAG on the query `"What are the liquidity coverage ratio requirements?"` — retrieve 5 chunks, critique each for relevance, filter, generate, critique answer quality. Print the final answer and a summary of what was kept vs filtered.
**Expected output**: The 5 retrieved chunks with their Yes/No relevance verdicts, the filtered set that reached generation, the generated answer with cited excerpts, and the answer quality verdict.

In [ ]:
QUERY = 'What are the liquidity coverage ratio requirements?'

print(f'Query: {QUERY!r}')
print()

t0 = time.perf_counter()
result = self_rag(QUERY, k=5)
total_ms = (time.perf_counter() - t0) * 1000

# ── Retrieval + relevance critique summary ────────────────────────────────────
print(f'Step 1–2: Retrieved {len(result["retrieved"])} chunks → critiqued for relevance')
print()
for i, (doc, cr) in enumerate(result['relevance_critiques'], 1):
    marker = '✓ kept   ' if cr.passed else '✗ filtered'
    preview = doc.page_content[:70].replace('\n', ' ')
    print(f'  [{i}] {cr.verdict:<3}  {marker}  {preview!r}...')
    print(f'       Reason: {cr.rationale}')
    print()

n_kept     = len(result['relevant_chunks'])
n_filtered = len(result['retrieved']) - n_kept
print(f'Step 3: Filtered — {n_kept} kept, {n_filtered} discarded')
print()

# ── Generated answer ──────────────────────────────────────────────────────────
print('Step 4: Generated answer (from relevant chunks only):')
print('=' * 65)
print(result['answer'])
print('=' * 65)
print()

# ── Answer quality critique ───────────────────────────────────────────────────
cr = result['answer_critique']
if cr:
    marker = '✓ passed' if cr.passed else '✗ failed'
    print(f'Step 5: Answer quality critique — {cr.verdict}  {marker}')
    print(f'        Reason: {cr.rationale}')
    print()

print(f'Final status: {result["final_status"]}')
print(f'Total time:   {total_ms:.0f} ms  ({len(result["retrieved"])} relevance calls + generation + quality critique)')

## Cell 5: Inspect — Critique Outputs and Filtering Decisions
**What this demonstrates**: Expose the full critique trace from Cell 4 — show the complete text of each filtered-out chunk with its rejection reason, the full relevant chunks that reached generation, and the answer quality rationale. This is the observability layer that makes Self-RAG's decisions auditable.
**Expected output**: Full text of each discarded chunk with explanation, full text of each kept chunk, and a structured critique summary showing what the Self-RAG loop rejected and why.

In [ ]:
# ── Show filtered-out chunks in full ──────────────────────────────────────────
filtered_pairs = [(doc, cr) for doc, cr in result['relevance_critiques'] if not cr.passed]

print('── FILTERED CHUNKS (verdict: No) ────────────────────────────────────────────')
if filtered_pairs:
    for i, (doc, cr) in enumerate(filtered_pairs, 1):
        print(f'  Filtered [{i}]')
        print(f'  Reason: {cr.rationale}')
        print(f'  Content (first 200 chars):')
        print(f'    {doc.page_content[:200].replace(chr(10), " ")}...')
        print()
    print(f'  These {len(filtered_pairs)} chunk(s) were retrieved on cosine similarity but rejected')
    print('  by the relevance critique. Without Self-RAG, they would have entered')
    print('  the generation context and potentially contaminated the answer.')
else:
    print('  All retrieved chunks passed relevance — no filtering occurred.')
    print('  Try a broader query to observe filtering in action.')

# ── Show kept chunks in full ──────────────────────────────────────────────────
print()
print('── KEPT CHUNKS (verdict: Yes) ───────────────────────────────────────────────')
kept_pairs = [(doc, cr) for doc, cr in result['relevance_critiques'] if cr.passed]
for i, (doc, cr) in enumerate(kept_pairs, 1):
    print(f'  Kept [{i}]  Reason: {cr.rationale}')
    print(f'  Content:')
    # Print full chunk text, indented
    for line in doc.page_content.split('\n')[:8]:
        print(f'    {line}')
    if len(doc.page_content.split('\n')) > 8:
        print('    ...')
    print()

# ── Answer quality critique in full ───────────────────────────────────────────
print('── ANSWER QUALITY CRITIQUE ──────────────────────────────────────────────────')
cr = result['answer_critique']
if cr:
    print(f'  Verdict:   {cr.verdict}')
    print(f'  Reason:    {cr.rationale}')
    print(f'  Passed:    {cr.passed}')
else:
    print('  No answer critique — abstained at relevance filtering stage.')

# ── Critique decision summary ─────────────────────────────────────────────────
print()
print('── CRITIQUE DECISION SUMMARY ────────────────────────────────────────────────')
print(f'  Chunks retrieved:         {len(result["retrieved"])}')
print(f'  Chunks passed ISREL:      {len(result["relevant_chunks"])}')
print(f'  Chunks filtered out:      {len(result["retrieved"]) - len(result["relevant_chunks"])}')
print(f'  Answer quality (ISSUP):   {cr.verdict if cr else "N/A"}')
print(f'  Final status:             {result["final_status"]}')
print()
print('This summary — stored alongside the answer — is the Self-RAG audit trail.')

## Cell 6: Fintech — Regulatory Interpretation with Verification
**What this demonstrates**: Self-RAG on a compliance interpretation query — `"What capital buffers must a bank maintain above minimum CET1 requirements?"` This query spans multiple Basel III articles (conservation buffer, countercyclical buffer), testing whether the critique loop correctly accepts multi-section relevant chunks and rejects unrelated sections. Demonstrates the abstain path by also running a query the corpus cannot answer.
**Expected output**: Grounded answer with buffer requirements cited from source, critique trace showing multi-section acceptance, and the abstain response for an out-of-scope query.

In [ ]:
# ── Compliance query: capital buffers above minimum CET1 ──────────────────────
# Tests ISREL on a query that spans two articles:
#   Article 1 (capital conservation buffer: +2.5%) and
#   Article 2 (countercyclical buffer: up to +2.5%)
# Chunks about leverage ratio or LCR should be filtered out.
BUFFER_QUERY = 'What capital buffers must a bank maintain above minimum CET1 requirements?'

print(f'Compliance query: {BUFFER_QUERY!r}')
print()

t0 = time.perf_counter()
buffer_result = self_rag(BUFFER_QUERY, k=5)
buffer_ms = (time.perf_counter() - t0) * 1000

# ── Relevance critique trace ──────────────────────────────────────────────────
print('ISREL critique results:')
for i, (doc, cr) in enumerate(buffer_result['relevance_critiques'], 1):
    marker = '✓' if cr.passed else '✗'
    preview = doc.page_content[:60].replace('\n', ' ')
    print(f'  {marker} [{i}] {cr.verdict:<3}  {preview!r}...')
    print(f'         {cr.rationale}')

print()
print(f'  → {len(buffer_result["relevant_chunks"])}/{len(buffer_result["retrieved"])} chunks passed to generation')
print()

# ── Grounded answer ───────────────────────────────────────────────────────────
cr_buf = buffer_result['answer_critique']
print('=' * 65)
print('REGULATORY INTERPRETATION — Capital Buffer Requirements')
print('=' * 65)
print(buffer_result['answer'])
print('=' * 65)
print()
if cr_buf:
    print(f'Answer quality verdict: {cr_buf.verdict}  |  {cr_buf.rationale}')
print(f'Status: {buffer_result["final_status"]}  |  {buffer_ms:.0f} ms')

# ── Abstain path: query the corpus cannot answer ──────────────────────────────
print()
print('─' * 65)
print('ABSTAIN PATH DEMONSTRATION')
print('─' * 65)
# This query is outside the corpus — no Basel III article covers IFRS 9 in this excerpt
OUT_OF_SCOPE = 'What are the IFRS 9 expected credit loss model requirements for stage 2 classification?'
print(f'Out-of-scope query: {OUT_OF_SCOPE!r}')
print()

t1 = time.perf_counter()
oos_result = self_rag(OUT_OF_SCOPE, k=5)
oos_ms = (time.perf_counter() - t1) * 1000

print('ISREL verdicts:')
for i, (doc, cr) in enumerate(oos_result['relevance_critiques'], 1):
    print(f'  {"✓" if cr.passed else "✗"} [{i}] {cr.verdict}  — {cr.rationale[:70]}')

print()
print(f'Chunks passed: {len(oos_result["relevant_chunks"])}/{len(oos_result["retrieved"])}')
print()
print('SYSTEM RESPONSE:')
print(f'  {oos_result["answer"]}')
print()
print(f'Status: {oos_result["final_status"]}  |  {oos_ms:.0f} ms')
print()
print('The abstain response is the hallucination prevention mechanism.')
print('In production: route to escalation, fallback search, or human review.')